# 07 - Pelabelan Topik (Human-in-the-Loop)

## Tujuan
Memberi nama yang bermakna pada 8 topik hasil LDA melalui analisis
interpretatif terhadap kata kunci teratas, lalu mengunci label dengan
fingerprint SHA-256.

## Asal (Input)
- `_fix_final/top_words_final.csv` (dari Notebook 08)
- `model/lda_model.gensim`

## Keluaran (Output)
`_label_work/labels.json` - 8 label final + fingerprint SHA-256

## Mengapa Human-in-the-Loop?
LDA menghasilkan topik sebagai distribusi kata, tanpa nama. topping
seperti "variabel, teknologi, model, kualitas, sukses, monitoring..."
tidak langsung bermakna bagi pembaca. Needed: analis domain yang
menghubungkan kata-kata itu ke phenomenon di Program Studi.

Sesuai teori (Lau et al., 2011; Kozlowski et al., 2021), pelabelan
manual post-hoc adalah praktik standar dalam topic modeling.

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=asymmetric, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Langkah 1 - Tampilkan Kata Kunci per Topik

In [2]:
tw = pd.read_csv(C.TOP_WORDS_FINAL)
for t in range(1, 9):
    words = tw[tw['topik_id']==t].nsmallest(10, 'rank')['word'].tolist()
    print(f'  Topik {t}: ' + ', '.join(words))
    print()

  Topik 1: variabel, teknologi, model, kualitas, tahu, sukses, monitoring, nikah, audit, kualitas_layan

  Topik 2: pegawai, publik, extreme_programming, administrasi, pengaruh, variabel, produk, spk, jual, admin

  Topik 3: model, klasifikasi, siakad, kualitas, kategori, identifikasi, teknologi, kualitas_layan, atribut, faktor

  Topik 4: puas_guna, puas, eucs, ease_use, model_end, ancam, computing_satisfaction, end_user, variabel, tracer_study

  Topik 5: model, risiko, teknologi, jual, tahu, usability, indonesia, lapor, manajemen, variabel

  Topik 6: kualitas, bimbing, variabel, manajemen, learning, karakteristik, tahu, sewa, ajar, iso_iec

  Topik 7: extreme_programming, kelola_data, manajemen, lapor, uang, model, kasus, didik, usaha, mesan

  Topik 8: bimbing_konseling, siswa, langgan, usaha, aman, audit, lapor, tahu, daftar, alumni

## Langkah 2 - Lihat Label yang Disetujui

In [3]:
import json
labels = json.load(open(C.LABELS_JSON, encoding='utf-8'))
for item in labels:
    print(f"  Topik {item['topic_id']}: {item['label']}")
    print(f"    Keyakinan : {item['tingkat_kepercayaan']}")
    print(f"    Deskripsi : {item['deskripsi'][:75]}")

  Topik 1: Audit TI & Evaluasi Kesuksesan SI
    Keyakinan : Sedang
    Deskripsi : Topik ini memuat skripsi seputar audit tata kelola IT, evaluasi kesuksesan/
  Topik 2: Pengembangan & Analisis Sistem Heterogen
    Keyakinan : Rendah
    Deskripsi : Topik ini berisi kelompok skripsi heterogen yang mencakup pengembangan sist
  Topik 3: Analisis Kualitas Layanan & Risiko SI
    Keyakinan : Sedang
    Deskripsi : Topik ini berfokus pada analisis kualitas layanan aplikasi/SIAKAD, evaluasi
  Topik 4: Kepuasan Pengguna Akhir (EUCS & TAM)
    Keyakinan : Tinggi
    Deskripsi : Topik ini secara spesifik mengelompokkan penelitian evaluasi kepuasan pengg
  Topik 5: Manajemen Risiko TI & Analitik Data
    Keyakinan : Sedang
    Deskripsi : Topik ini memuat studi manajemen risiko teknologi informasi serta penerapan
  Topik 6: Pengembangan & Perencanaan Strategis SI
    Keyakinan : Sedang
    Deskripsi : Topik ini mencakup skripsi mengenai perencanaan strategis sistem informasi,
  Topik 7: Rancang

## Langkah 3 - Validasi Fingerprint terhadap Model

In [4]:
from src.topic_labels import validate_labels
validate_labels(str(C.LDA_MODEL), str(C.LABELS_JSON))
print('[OK] Seluruh fingerprint label cocok dengan model LDA')

[OK] Seluruh fingerprint label cocok dengan model LDA

## Langkah 4 - Ekspor Label ke CSV

In [5]:
from src.labeling import run_auto_labeling
run_auto_labeling()

✅ Topic labels CSV successfully created at -> C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main\model\topic_labels.csv

## Ringkasan Fase 07 - Labeling

Delapan topik telah diberi label melalui analisis interpretatif dan
dikunci dengan fingerprint SHA-256. Mekanisme ini menjamin label tidak
pernah "melayang" ke model yang berbeda karena setiap perubahan model
akan menyebabkan validasi gagal.

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `_label_work/labels.json` | Notebook 07 | Notebook 04, 06, 08, app |